
# 10. HICP Energy aggregate forecast

This notebook is the aggregation layer above the seven ECB-style energy BVAR equations.

It does **not** re-estimate the BVARs. It:

1. validates the historical HICP aggregation identities;
2. reloads the saved posterior predictive paths from notebooks 03–09;
3. maps the weekly petrol, diesel and liquid-fuel forecasts into monthly HICP component indices;
4. builds the energy-relevant **car-fuels** sub-aggregate;
5. aggregates the six model components into **HICP Energy** with annual Laspeyres-type weights and December chain links;
6. computes draw-wise YoY inflation and exactly additive contributions;
7. optionally applies tax scenarios ex post;
8. saves one aggregate result store for the future dashboard.

### Important statistical convention

The seven BVARs are estimated independently. Equal Gibbs draw numbers across two models do **not** form a joint posterior draw. The notebook therefore randomises the pairing of component paths explicitly. This is the maintained cross-model independence approximation for the unconditional aggregate forecast.

### Weekly tax data

The paper-faithful route is:

\[
P^{pre-tax}\rightarrow (VAT,EXC)\rightarrow P^{after-tax}\rightarrow HICP.
\]

If the current single-workbook WOB sheet does not expose the separate VAT/IDT series, `WEEKLY_TAX_MODE="auto"` uses an explicitly labelled empirical **pre-tax WOB → HICP** bridge for the baseline aggregate. It never pretends that this fallback is the paper's tax decomposition. Set `WEEKLY_TAX_MODE="strict"` to forbid that approximation.


In [1]:

from __future__ import annotations

from pathlib import Path
import hashlib
import json
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "processed").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())

MODEL_DIR_CANDIDATES = [
    PROJECT_ROOT / "src" / "model",
    PROJECT_ROOT,
    Path.cwd(),
]

REQUIRED_MODULES = (
    "energy_bvar_aggregate.py",
    "energy_bvar_io.py",
    "energy_bvar_gas.py",
    "energy_bvar_electricity.py",
    "energy_bvar_weekly_fuels.py",
    "energy_bvar_source_context.py",
)

MODEL_DIR = None
for candidate in MODEL_DIR_CANDIDATES:
    if all((candidate / name).is_file() for name in REQUIRED_MODULES):
        sys.path.insert(0, str(candidate))
        MODEL_DIR = candidate
        break

if MODEL_DIR is None:
    raise FileNotFoundError(
        "Place the canonical model files beside this notebook or in src/model/: "
        + ", ".join(REQUIRED_MODULES)
    )

from energy_bvar_io import load_energy_bvar_forecast
from energy_bvar_aggregate import (
    aggregate_component_draw_paths_laspeyres,
    aggregate_draw_paths_laspeyres,
    assert_reconstruction_suite,
    bridge_diagnostic_table,
    carry_last_index_path,
    draw_yoy_and_contributions,
    fit_price_index_bridge,
    historical_model_component_reconstruction,
    historical_reconstruction_suite,
    independent_draw_pairing,
    load_aggregation_inputs,
    price_paths_to_hicp_paths,
    summarise_draw_paths,
    weekly_paths_with_history_to_monthly_mean,
)
from energy_bvar_gas import (
    expand_semester_series as expand_gas_semester_series,
    load_gas_tax_context,
    reattribute_gas_taxes,
    validate_tax_round_trip,
)
from energy_bvar_electricity import (
    expand_semester_series as expand_electricity_semester_series,
    load_electricity_tax_context,
    reattribute_electricity_taxes,
    validate_electricity_tax_round_trip,
)
from energy_bvar_weekly_fuels import (
    load_weekly_tax_context,
    reattribute_weekly_taxes,
)
from energy_bvar_source_context import load_manifest_source_frame

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 190)
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

print(f"Project root: {PROJECT_ROOT}")
print(f"Model modules: {MODEL_DIR}")


Project root: C:\Users\kelyan.gane\Project\Python\Bayesian_model\bvar-energy
Model modules: C:\Users\kelyan.gane\Project\Python\Bayesian_model\bvar-energy\src\model



## 1. Configuration and forecast-store preflight

`FORECAST_NAME="unconditional"` is the baseline Notebook 10 contract.

The component notebooks must save their predictive paths once with the updated `energy_bvar_io.py`. The required two calls are:

```python
saved_result_paths = save_energy_bvar_result(result, RESULTS_ROOT)
save_energy_bvar_forecast(
    forecast_unconditional,
    saved_result_paths["directory"],
    "unconditional",
)
```

Do this in notebooks 03–09 after the forecast object has been created. It does **not** alter the model; it only persists the already-created posterior predictive paths.


In [2]:

VINTAGE = None                 # e.g. "20260808"; None -> latest valid processed vintage
FORECAST_NAME = "unconditional"
RESULTS_ROOT = PROJECT_ROOT / "results"

N_AGG_DRAWS = 500
PAIRING_SEED = 2026

# "auto"   : use official WOB VAT/IDT when available; otherwise explicit
#            empirical pre-tax bridge fallback for the baseline.
# "strict" : official WOB VAT/IDT are mandatory.
# "empirical_pre_tax_bridge" : deliberately force the fallback.
WEEKLY_TAX_MODE = "auto"

SAVE_AGGREGATE_RESULT = True

# Optional ex-post tax scenarios. None means the paper baseline:
# latest published tax state carried forward.
#
# Example for gas/electricity:
# {
#     "start_date": "2026-10-01",
#     "vat_percent": 21.0,
# }
#
# An excise scenario must also specify the exact unit expected by the adapter.
TAX_SCENARIOS = {
    "petrol": None,
    "diesel": None,
    "liquid_fuels": None,
    "gas": None,
    "electricity": None,
}

MODEL_IDS = {
    "gas": "gas",
    "electricity": "electricity",
    "heat_energy": "heat_energy",
    "solid_fuels": "solid_fuels",
    "petrol": "car_fuels_petrol",
    "diesel": "car_fuels_diesel",
    "liquid_fuels": "liquid_fuels",
}

DATASET_FILES = {
    "gas": "gas_monthly.csv",
    "electricity": "electricity_monthly.csv",
    "heat_energy": "heat_energy_monthly.csv",
    "solid_fuels": "solid_fuels_monthly.csv",
    "petrol": "car_fuels_weekly.csv",
    "diesel": "car_fuels_weekly.csv",
    "liquid_fuels": "liquid_fuels_weekly.csv",
}

PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

if VINTAGE is None:
    candidates = sorted(
        p for p in PROCESSED_ROOT.iterdir()
        if p.is_dir()
        and (p / "manifest.json").is_file()
        and (p / "hicp_indices_monthly.csv").is_file()
        and (p / "hicp_weights_annual.csv").is_file()
    )
    if not candidates:
        raise FileNotFoundError(
            f"No v10 processed vintage with HICP aggregation sidecars found below {PROCESSED_ROOT}."
        )
    PROCESSED_DIR = candidates[-1]
    VINTAGE = PROCESSED_DIR.name
else:
    PROCESSED_DIR = PROCESSED_ROOT / str(VINTAGE)

if not PROCESSED_DIR.is_dir():
    raise FileNotFoundError(PROCESSED_DIR)

for name, filename in DATASET_FILES.items():
    if not (PROCESSED_DIR / filename).is_file():
        raise FileNotFoundError(
            f"{name}: required processed dataset is missing: {PROCESSED_DIR / filename}"
        )


def latest_forecast_store(model_id: str, vintage: str, forecast_name: str) -> Path | None:
    base = RESULTS_ROOT / model_id / str(vintage)
    if not base.is_dir():
        return None
    stores = [
        run / "forecasts" / forecast_name
        for run in base.iterdir()
        if run.is_dir()
        and (run / "forecasts" / forecast_name / "forecast_metadata.json").is_file()
        and (run / "forecasts" / forecast_name / "forecast_draws.npz").is_file()
    ]
    if not stores:
        return None
    return max(stores, key=lambda path: path.stat().st_mtime_ns)


FORECAST_STORES = {
    name: latest_forecast_store(model_id, VINTAGE, FORECAST_NAME)
    for name, model_id in MODEL_IDS.items()
}

preflight = pd.DataFrame([
    {
        "component_model": name,
        "model_id": MODEL_IDS[name],
        "processed_dataset": DATASET_FILES[name],
        "forecast_store": None if path is None else str(path),
        "ready": path is not None,
    }
    for name, path in FORECAST_STORES.items()
]).set_index("component_model")

display(preflight)

missing_forecasts = preflight.index[~preflight["ready"]].tolist()
if missing_forecasts:
    raise FileNotFoundError(
        "Notebook 10 cannot aggregate marginal medians: it needs the posterior "
        "predictive draws. Missing saved forecast stores for "
        f"{missing_forecasts}. In notebooks 03–09, import "
        "save_energy_bvar_forecast from energy_bvar_io, save the model result, "
        "then save forecast_unconditional under the name 'unconditional'. "
        "If those notebook kernels are still alive, only the save cells need to "
        "be run; no re-estimation is required."
    )

print(f"Processed vintage: {VINTAGE}")
print(f"Processed directory: {PROCESSED_DIR}")


FileNotFoundError: No v10 processed vintage with HICP aggregation sidecars found below C:\Users\kelyan.gane\Project\Python\Bayesian_model\bvar-energy\data\processed.


## 2. Historical aggregation gate

No forecast is accepted before the deterministic history closes.

The notebook verifies:

\[
CP045 = CP0451+CP0452+CP0453+CP0454+CP0455,
\]

the long-history special-aggregate split,

\[
NRG = ELC\_GAS+FUEL,
\]

and, from the ECOICOP-v2 break onward, the detailed transport and Energy decompositions.

It then builds the exact six-model historical decomposition used later for draw-wise contributions.


In [ ]:

aggregation_inputs = load_aggregation_inputs(PROCESSED_DIR)
indices = aggregation_inputs["indices"]
weights = aggregation_inputs["weights"]

historical_validation = historical_reconstruction_suite(PROCESSED_DIR)
model_history = historical_model_component_reconstruction(PROCESSED_DIR)

# Always show every cumulative and annual re-anchored test before the hard gate.
display(historical_validation["summary"])
display(model_history["summary"].to_frame("value"))

assert_reconstruction_suite(historical_validation)
if not bool(model_history["summary"]["pass"]):
    raise AssertionError(
        "Six-model historical HICP Energy reconstruction failed: "
        + str(model_history["summary"].to_dict())
    )

published_energy = indices["hicp_energy"].dropna()
reconstructed_energy = model_history["energy"]["index"]

comparison = pd.concat(
    [
        published_energy.rename("published HICP Energy"),
        reconstructed_energy.rename("six-model reconstruction"),
    ],
    axis=1,
).dropna()

fig, ax = plt.subplots(figsize=(11, 4.5))
shown = comparison.iloc[-84:]
ax.plot(shown.index, shown.iloc[:, 0], label=shown.columns[0])
ax.plot(shown.index, shown.iloc[:, 1], linestyle="--", label=shown.columns[1])
ax.set_title("Historical HICP Energy reconstruction")
ax.set_ylabel("Index")
ax.grid(alpha=0.22)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()



## 3. Load the seven posterior predictive stores

Every store contains the **full draw-by-date level paths**, not only a median and fan quantiles.


In [ ]:

forecasts = {
    name: load_energy_bvar_forecast(path)
    for name, path in FORECAST_STORES.items()
}

forecast_contract_rows = []
for name, forecast in forecasts.items():
    paths = np.asarray(forecast["level_paths"], dtype=float)
    dates = pd.DatetimeIndex(forecast["path_dates"])
    forecast_contract_rows.append({
        "model": name,
        "frequency": forecast["frequency"],
        "draws": paths.shape[0],
        "path_start": dates.min(),
        "path_end": dates.max(),
        "future_start": pd.DatetimeIndex(forecast["future_dates"]).min(),
        "future_end": pd.DatetimeIndex(forecast["future_dates"]).max(),
        "tail_length": int(forecast["tail_length"]),
        "variables": " | ".join(forecast["variables"]),
    })

forecast_contract = pd.DataFrame(forecast_contract_rows).set_index("model")
display(forecast_contract)

assert forecasts["gas"]["frequency"] == "monthly"
assert forecasts["electricity"]["frequency"] == "monthly"
assert forecasts["heat_energy"]["frequency"] == "monthly"
assert forecasts["solid_fuels"]["frequency"] == "monthly"
assert forecasts["petrol"]["frequency"] == "weekly"
assert forecasts["diesel"]["frequency"] == "weekly"
assert forecasts["liquid_fuels"]["frequency"] == "weekly"



## 4. Gas and electricity: exact tax re-attribution

For these two components the processed-vintage manifest already contains the estimated \(\gamma\), while Eurostat supplies VAT and excise. The historical round trip

\[
HICP \rightarrow P^{pre-tax}\rightarrow HICP
\]

must remain exact before forecast paths are used.


In [ ]:

gas_dataset = PROCESSED_DIR / DATASET_FILES["gas"]
electricity_dataset = PROCESSED_DIR / DATASET_FILES["electricity"]

gas_tax_context = load_gas_tax_context(gas_dataset)
electricity_tax_context = load_electricity_tax_context(electricity_dataset)

display(validate_tax_round_trip(gas_tax_context).to_frame("gas"))
display(validate_electricity_tax_round_trip(electricity_tax_context).to_frame("electricity"))


def _scenario_values(value, dates: pd.DatetimeIndex, label: str) -> np.ndarray:
    if isinstance(value, pd.Series):
        series = value.astype(float).copy()
        series.index = pd.DatetimeIndex(series.index).to_period("M").to_timestamp(how="start")
        if series.index.has_duplicates:
            raise ValueError(f"{label}: duplicate scenario months.")
        array = series.reindex(dates).to_numpy(dtype=float)
    else:
        array = np.asarray(value, dtype=float)
        if array.ndim == 0:
            array = np.repeat(float(array), len(dates))
    if array.ndim != 1 or len(array) != len(dates):
        raise ValueError(f"{label}: expected scalar, dated Series or length {len(dates)}.")
    if not np.all(np.isfinite(array)):
        raise ValueError(f"{label}: scenario contains non-finite values.")
    return array


def _apply_monthly_scenario(
    baseline_vat: pd.Series,
    baseline_excise: pd.Series,
    scenario: dict | None,
    *,
    excise_unit: str,
) -> tuple[pd.Series, pd.Series]:
    vat = baseline_vat.astype(float).copy()
    excise = baseline_excise.astype(float).copy()
    if scenario is None:
        return vat, excise

    scenario = dict(scenario)
    if "start_date" not in scenario:
        raise ValueError("Monthly tax scenario requires 'start_date'.")
    if "vat_percent" not in scenario and "excise" not in scenario:
        raise ValueError("Monthly tax scenario must change VAT and/or excise.")

    start = pd.Timestamp(scenario["start_date"]).to_period("M").to_timestamp(how="start")
    active_dates = vat.index[vat.index >= start]
    if len(active_dates) == 0:
        raise ValueError(f"Tax scenario starts after the forecast path: {start.date()}.")

    if "vat_percent" in scenario:
        values = _scenario_values(scenario["vat_percent"], active_dates, "vat_percent")
        if np.any((values < 0) | (values > 100)) or np.any((values > 0) & (values < 1)):
            raise ValueError("VAT must be supplied in percentage points, e.g. 21.0.")
        vat.loc[active_dates] = values

    if "excise" in scenario:
        supplied = scenario.get("excise_unit")
        if supplied is None:
            raise ValueError(f"Excise scenario requires excise_unit={excise_unit!r}.")
        norm = lambda x: "".join(str(x).lower().split())
        if norm(supplied) != norm(excise_unit):
            raise ValueError(
                f"Excise unit mismatch: expected {excise_unit!r}, got {supplied!r}."
            )
        excise.loc[active_dates] = _scenario_values(
            scenario["excise"], active_dates, "excise"
        )

    return vat, excise


def monthly_pretax_forecast_to_hicp(
    forecast: dict,
    *,
    target_variable: str,
    tax_context: dict,
    expand_function,
    reattribute_function,
    tax_scenario: dict | None,
) -> dict:
    dates = pd.DatetimeIndex(forecast["path_dates"], name="date")
    variables = list(forecast["variables"])
    if target_variable not in variables:
        raise KeyError(f"{target_variable!r} absent from {variables}.")

    j = variables.index(target_variable)
    pre_tax = np.asarray(forecast["level_paths"], dtype=float)[:, :, j]

    baseline_vat = expand_function(tax_context["vat_percent"], dates)
    baseline_excise = expand_function(tax_context["excise"], dates)

    scenario_vat, scenario_excise = _apply_monthly_scenario(
        baseline_vat,
        baseline_excise,
        tax_scenario,
        excise_unit=str(tax_context["excise_unit"]),
    )

    baseline = reattribute_function(
        pre_tax,
        float(tax_context["gamma"]),
        baseline_vat.to_numpy(dtype=float)[None, :],
        baseline_excise.to_numpy(dtype=float)[None, :],
    )
    scenario = reattribute_function(
        pre_tax,
        float(tax_context["gamma"]),
        scenario_vat.to_numpy(dtype=float)[None, :],
        scenario_excise.to_numpy(dtype=float)[None, :],
    )
    return {
        "dates": dates,
        "baseline": np.asarray(baseline, dtype=float),
        "scenario": np.asarray(scenario, dtype=float),
    }


gas_hicp = monthly_pretax_forecast_to_hicp(
    forecasts["gas"],
    target_variable="gas_pre_tax",
    tax_context=gas_tax_context,
    expand_function=expand_gas_semester_series,
    reattribute_function=reattribute_gas_taxes,
    tax_scenario=TAX_SCENARIOS["gas"],
)

electricity_hicp = monthly_pretax_forecast_to_hicp(
    forecasts["electricity"],
    target_variable="electricity_pre_tax",
    tax_context=electricity_tax_context,
    expand_function=expand_electricity_semester_series,
    reattribute_function=reattribute_electricity_taxes,
    tax_scenario=TAX_SCENARIOS["electricity"],
)

print("Gas/electricity HICP paths ready.")



## 5. Heat energy and solid fuels

These BVAR targets are already HICP indices. No tax bridge is required.


In [ ]:

def direct_hicp_paths(forecast: dict, target: str) -> dict:
    variables = list(forecast["variables"])
    if target not in variables:
        raise KeyError(f"{target!r} absent from {variables}.")
    j = variables.index(target)
    return {
        "dates": pd.DatetimeIndex(forecast["path_dates"], name="date"),
        "baseline": np.asarray(forecast["level_paths"], dtype=float)[:, :, j],
        "scenario": np.asarray(forecast["level_paths"], dtype=float)[:, :, j],
    }


heat_hicp = direct_hicp_paths(
    forecasts["heat_energy"],
    "hicp_heat_energy",
)
solid_hicp = direct_hicp_paths(
    forecasts["solid_fuels"],
    "hicp_solid_fuels",
)

print("Heat-energy and solid-fuels HICP paths ready.")



## 6. Petrol, diesel and liquid fuels: weekly price → monthly HICP bridge

The preferred route uses WOB after-tax prices reconstructed from separate VAT and IDT series.

The aggregation bridge deliberately follows the **absolute-change** logic of the STIP models:

\[
\Delta I_t^{HICP}
=
\beta\,\Delta P_t^{WOB}+u_t,
\]

with **no applied intercept**. Hence a constant consumer-price path implies a constant HICP path exactly:

\[
\Delta P_t^{WOB}=0
\quad\Rightarrow\quad
\Delta I_t^{HICP}=0.
\]

For diagnosis only, the code also estimates an unrestricted intercept and reports its \(t\)-statistic. A material intercept would stop the bridge rather than inject deterministic inflation into the forecast. The slope is also checked across two half-samples.

This is preferable to a log-growth bridge here because the paper deliberately models petrol, diesel and liquid-fuel consumer prices in **absolute differences**: per-unit excise duties and broadly stable refining/distribution margins make percentage-change relationships less stable. (ECB Working Paper 3062, Section 2.2).

When the current workbook exposes only WOB `WTAX` and `NTAX`, `auto` mode falls back to a direct **pre-tax WOB → HICP absolute-change bridge**. That baseline approximation is recorded in the final metadata, and weekly tax scenarios remain unavailable until VAT/IDT are restored.


In [ ]:

WEEKLY_MODELS = ("petrol", "diesel", "liquid_fuels")
WEEKLY_HICP_COLUMNS = {
    "petrol": "hicp_petrol",
    "diesel": "hicp_diesel",
    "liquid_fuels": "hicp_liquid_fuels",
}
WEEKLY_TARGETS = {
    "petrol": "wob_petrol_pre_tax",
    "diesel": "wob_diesel_pre_tax",
    "liquid_fuels": "wob_heating_oil_pre_tax",
}
WEEKLY_DATASETS = {
    "petrol": PROCESSED_DIR / "car_fuels_weekly.csv",
    "diesel": PROCESSED_DIR / "car_fuels_weekly.csv",
    "liquid_fuels": PROCESSED_DIR / "liquid_fuels_weekly.csv",
}

EC_RENAME = {
    "EUR_price_with_tax_euro95": "wob_petroleum_cpr_wtax",
    "EUR_price_wo_tax_euro95": "wob_petroleum_cpr_ntax",
    "EUR_price_with_tax_diesel": "wob_diesel_cpr_wtax",
    "EUR_price_wo_tax_diesel": "wob_diesel_cpr_ntax",
    "EUR_price_with_tax_heating_oil": "wob_gas_cpr_wtax",
    "EUR_price_wo_tax_heating_oil": "wob_gas_cpr_ntax",
}
FALLBACK_PRETAX_COLUMNS = {
    "petrol": "wob_petroleum_cpr_ntax",
    "diesel": "wob_diesel_cpr_ntax",
    "liquid_fuels": "wob_gas_cpr_ntax",
}

weekly_tax_contexts = {}
weekly_tax_errors = {}
weekly_monthly_coverage = {}

if WEEKLY_TAX_MODE not in {"auto", "strict", "empirical_pre_tax_bridge"}:
    raise ValueError(
        "WEEKLY_TAX_MODE must be 'auto', 'strict' or 'empirical_pre_tax_bridge'."
    )

if WEEKLY_TAX_MODE != "empirical_pre_tax_bridge":
    for model in WEEKLY_MODELS:
        try:
            weekly_tax_contexts[model] = load_weekly_tax_context(
                WEEKLY_DATASETS[model],
                model=model,
            )
        except (FileNotFoundError, KeyError, ValueError) as exc:
            weekly_tax_errors[model] = str(exc)

if WEEKLY_TAX_MODE == "strict" and weekly_tax_errors:
    display(pd.Series(weekly_tax_errors, name="strict WOB tax-context error"))
    raise RuntimeError(
        "Strict weekly tax mode requires the separate WOB VAT and IDT series. "
        "The current workbook does not expose a complete weekly tax block."
    )

if WEEKLY_TAX_MODE == "empirical_pre_tax_bridge":
    WEEKLY_TAX_MODE_EFFECTIVE = "empirical_pre_tax_bridge"
elif weekly_tax_errors:
    WEEKLY_TAX_MODE_EFFECTIVE = "empirical_pre_tax_bridge"
    warnings.warn(
        "Official weekly WOB VAT/IDT context is incomplete. Notebook 10 is "
        "using the explicitly labelled empirical pre-tax WOB -> HICP bridge "
        "for petrol, diesel and liquid fuels. This is not the paper-faithful "
        "tax decomposition.",
        RuntimeWarning,
    )
else:
    WEEKLY_TAX_MODE_EFFECTIVE = "official_wob_tax_reattribution"

print(f"Requested weekly tax mode: {WEEKLY_TAX_MODE}")
print(f"Effective weekly tax mode: {WEEKLY_TAX_MODE_EFFECTIVE}")
if weekly_tax_errors:
    display(pd.Series(weekly_tax_errors, name="WOB tax-context diagnostic"))


def monthly_mean_series(series: pd.Series) -> pd.Series:
    series = series.astype(float).dropna().sort_index()
    grouped = series.groupby(series.index.to_period("M")).mean()
    grouped.index = grouped.index.to_timestamp(how="start")
    grouped.index = pd.DatetimeIndex(grouped.index, name="date")
    return grouped


def extend_monthly_price_paths_from_last_hicp(
    forecast_monthly_paths: np.ndarray,
    forecast_monthly_dates: pd.DatetimeIndex,
    historical_weekly_price: pd.Series,
    hicp_history: pd.Series,
) -> tuple[np.ndarray, pd.DatetimeIndex, float, float]:
    """Prepend fully observed monthly WOB prices between the HICP edge and forecast path."""
    hicp = hicp_history.astype(float).dropna().sort_index()
    if hicp.empty:
        raise ValueError("HICP bridge history is empty.")
    last_hicp_date = pd.Timestamp(hicp.index[-1]).to_period("M").to_timestamp(how="start")
    anchor_hicp = float(hicp.iloc[-1])

    historical_monthly = monthly_mean_series(historical_weekly_price)
    if last_hicp_date not in historical_monthly.index:
        raise ValueError(
            f"WOB monthly price is unavailable at the HICP anchor {last_hicp_date.date()}."
        )
    anchor_price = float(historical_monthly.loc[last_hicp_date])

    dates = pd.DatetimeIndex(forecast_monthly_dates, name="date")
    target_dates = pd.date_range(
        last_hicp_date + pd.offsets.MonthBegin(1),
        dates[-1],
        freq="MS",
        name="date",
    )
    n_draws = forecast_monthly_paths.shape[0]
    columns = []
    for date in target_dates:
        if date in dates:
            columns.append(
                np.asarray(
                    forecast_monthly_paths[:, dates.get_loc(date)],
                    dtype=float,
                )
            )
        elif date in historical_monthly.index:
            columns.append(
                np.full(n_draws, float(historical_monthly.loc[date]), dtype=float)
            )
        else:
            raise ValueError(
                f"No observed or forecast WOB monthly price is available for {date.date()}."
            )

    return np.column_stack(columns), target_dates, anchor_price, anchor_hicp


# Load the always-available aggregate WOB price history once. It is also the
# transparent fallback source if separate VAT/IDT are absent.
wob_price_frame, wob_price_source = load_manifest_source_frame(
    PROCESSED_DIR / "car_fuels_weekly.csv",
    "european_commission",
    simple_rename=EC_RENAME,
)
wob_price_frame.index = pd.DatetimeIndex(
    wob_price_frame.index
).to_period("W-SUN").start_time
wob_price_frame.index.name = "date"

bridges = {}
weekly_hicp_paths = {}
weekly_bridge_basis = {}

for model in WEEKLY_MODELS:
    forecast = forecasts[model]
    target = WEEKLY_TARGETS[model]
    variables = list(forecast["variables"])
    j = variables.index(target)

    if WEEKLY_TAX_MODE_EFFECTIVE == "official_wob_tax_reattribution":
        context = weekly_tax_contexts[model]
        baseline_taxed = reattribute_weekly_taxes(
            forecast,
            context,
            tax_scenario=None,
        )
        scenario_taxed = reattribute_weekly_taxes(
            forecast,
            context,
            tax_scenario=TAX_SCENARIOS[model],
        )

        baseline_monthly, monthly_dates, baseline_coverage = weekly_paths_with_history_to_monthly_mean(
            baseline_taxed["post_tax_level_paths"],
            baseline_taxed["path_dates"],
            context["data"]["after_tax"],
            return_coverage=True,
        )
        scenario_monthly, scenario_dates, scenario_coverage = weekly_paths_with_history_to_monthly_mean(
            scenario_taxed["post_tax_level_paths"],
            scenario_taxed["path_dates"],
            context["data"]["after_tax"],
            return_coverage=True,
        )
        if not monthly_dates.equals(scenario_dates):
            raise RuntimeError(f"{model}: baseline/scenario monthly calendars differ.")
        if not baseline_coverage.equals(scenario_coverage):
            raise RuntimeError(f"{model}: baseline/scenario monthly coverage differs.")
        weekly_monthly_coverage[model] = baseline_coverage

        historical_price = context["data"]["after_tax"]
        weekly_bridge_basis[model] = "after-tax WOB price"

    else:
        if TAX_SCENARIOS[model] is not None:
            raise RuntimeError(
                f"{model}: a weekly tax scenario was requested, but separate WOB "
                "VAT/IDT are unavailable. Restore those source series or use no "
                "weekly scenario."
            )

        raw_pre_tax = wob_price_frame[FALLBACK_PRETAX_COLUMNS[model]].astype(float)
        pre_tax_paths = np.asarray(
            forecast["level_paths"], dtype=float
        )[:, :, j]
        baseline_monthly, monthly_dates, baseline_coverage = weekly_paths_with_history_to_monthly_mean(
            pre_tax_paths,
            forecast["path_dates"],
            raw_pre_tax,
            return_coverage=True,
        )
        weekly_monthly_coverage[model] = baseline_coverage
        scenario_monthly = baseline_monthly.copy()
        historical_price = raw_pre_tax
        weekly_bridge_basis[model] = "pre-tax WOB price — explicit fallback"

    hicp_history = indices[WEEKLY_HICP_COLUMNS[model]]
    bridge = fit_price_index_bridge(
        monthly_mean_series(historical_price),
        hicp_history,
        label=f"{model}: {weekly_bridge_basis[model]} -> HICP",
    )
    bridges[model] = bridge

    baseline_extended, bridge_dates, anchor_price, anchor_hicp = (
        extend_monthly_price_paths_from_last_hicp(
            baseline_monthly,
            monthly_dates,
            historical_price,
            hicp_history,
        )
    )
    scenario_extended, scenario_bridge_dates, _, _ = (
        extend_monthly_price_paths_from_last_hicp(
            scenario_monthly,
            monthly_dates,
            historical_price,
            hicp_history,
        )
    )
    if not bridge_dates.equals(scenario_bridge_dates):
        raise RuntimeError(f"{model}: baseline/scenario bridge calendars differ.")

    weekly_hicp_paths[model] = {
        "dates": bridge_dates,
        "baseline": price_paths_to_hicp_paths(
            baseline_extended,
            bridge_dates,
            bridge=bridge,
            anchor_price=anchor_price,
            anchor_hicp=anchor_hicp,
        ),
        "scenario": price_paths_to_hicp_paths(
            scenario_extended,
            bridge_dates,
            bridge=bridge,
            anchor_price=anchor_price,
            anchor_hicp=anchor_hicp,
        ),
    }

bridge_table = bridge_diagnostic_table(bridges)
bridge_table["price_basis"] = pd.Series(weekly_bridge_basis)
display(bridge_table)

print("Weekly-to-monthly coverage diagnostics:")
for model in WEEKLY_MODELS:
    print(f"\n{model}")
    display(weekly_monthly_coverage[model])

for model in WEEKLY_MODELS:
    print(
        f"{model:13s}: {weekly_hicp_paths[model]['dates'].min().date()} -> "
        f"{weekly_hicp_paths[model]['dates'].max().date()} | "
        f"{weekly_hicp_paths[model]['baseline'].shape[0]} draws"
    )



## 7. Build the energy-relevant car-fuels index

For HICP Energy the transport block excludes lubricants. The forecasted car-fuels basket is therefore

\[
\{\text{diesel},\ \text{petrol},\ \text{other transport fuels}\}.
\]

`Other transport fuels` has no dedicated BVAR in the paper suite. Its latest published HICP level is held constant over the short forecast horizon. Its raw HICP weight remains in the Laspeyres aggregation, so it is not silently dropped.


In [ ]:

def common_monthly_dates(objects: list[dict]) -> pd.DatetimeIndex:
    common = None
    for obj in objects:
        dates = pd.DatetimeIndex(obj["dates"], name="date")
        common = dates if common is None else common.intersection(dates)
    if common is None or len(common) == 0:
        raise ValueError("No common monthly aggregation dates.")
    common = pd.DatetimeIndex(common.sort_values(), name="date")
    expected = pd.date_range(common[0], common[-1], freq="MS", name="date")
    if not common.equals(expected):
        raise ValueError("The common monthly path contains an interior gap.")
    return common


def align_paths(obj: dict, key: str, dates: pd.DatetimeIndex) -> np.ndarray:
    source_dates = pd.DatetimeIndex(obj["dates"], name="date")
    positions = source_dates.get_indexer(dates)
    if (positions < 0).any():
        raise ValueError("Requested dates are absent from a component path.")
    return np.asarray(obj[key], dtype=float)[:, positions]


transport_dates = common_monthly_dates([
    weekly_hicp_paths["petrol"],
    weekly_hicp_paths["diesel"],
])

petrol_base = align_paths(weekly_hicp_paths["petrol"], "baseline", transport_dates)
diesel_base = align_paths(weekly_hicp_paths["diesel"], "baseline", transport_dates)
petrol_scenario = align_paths(weekly_hicp_paths["petrol"], "scenario", transport_dates)
diesel_scenario = align_paths(weekly_hicp_paths["diesel"], "scenario", transport_dates)

other_transport = carry_last_index_path(
    indices["hicp_other_transport_fuels"],
    transport_dates,
    n_draws=max(N_AGG_DRAWS, 1),
)

transport_base_unpaired = {
    "hicp_petrol": petrol_base,
    "hicp_diesel": diesel_base,
    "hicp_other_transport_fuels": other_transport,
}

transport_base_paired, transport_pair_indices = independent_draw_pairing(
    transport_base_unpaired,
    n_draws=N_AGG_DRAWS,
    seed=PAIRING_SEED,
)

transport_scenario_paired = {
    "hicp_petrol": petrol_scenario[transport_pair_indices["hicp_petrol"]],
    "hicp_diesel": diesel_scenario[transport_pair_indices["hicp_diesel"]],
    "hicp_other_transport_fuels": other_transport[
        transport_pair_indices["hicp_other_transport_fuels"]
    ],
}

transport_components = [
    "hicp_diesel",
    "hicp_petrol",
    "hicp_other_transport_fuels",
]
transport_weights = weights[transport_components]

transport_history_index = pd.concat(
    [
        pd.Series(
            [100.0],
            index=pd.DatetimeIndex([pd.Timestamp("2016-12-01")], name="date"),
        ),
        model_history["transport_energy"]["index"],
    ]
).sort_index()
transport_history_index.name = "car_fuels"

transport_component_history = indices[transport_components]

car_fuels_baseline = aggregate_component_draw_paths_laspeyres(
    transport_component_history,
    transport_base_paired,
    transport_dates,
    transport_weights,
    transport_history_index,
    components=transport_components,
)

car_fuels_scenario = aggregate_component_draw_paths_laspeyres(
    transport_component_history,
    transport_scenario_paired,
    transport_dates,
    transport_weights,
    transport_history_index,
    components=transport_components,
)

assert np.array_equal(
    car_fuels_baseline["weight_year_used"].to_numpy(),
    car_fuels_scenario["weight_year_used"].to_numpy(),
)

print(
    f"Car-fuels path: {transport_dates.min().date()} -> "
    f"{transport_dates.max().date()} | {N_AGG_DRAWS} matched draws"
)



## 8. Aggregate the six model components into HICP Energy

The six blocks are:

\[
\text{Car fuels},\quad
\text{Liquid fuels},\quad
\text{Gas},\quad
\text{Electricity},\quad
\text{Heat energy},\quad
\text{Solid fuels}.
\]

The final aggregation is performed **draw by draw**. Annual raw Eurostat weights are normalised only inside HICP Energy. If the horizon crosses into a year for which new weights have not yet been published, the latest available annual weight vector is carried forward and the weight year used is stored.


In [ ]:

component_objects = {
    "car_fuels": {
        "dates": transport_dates,
        "baseline": car_fuels_baseline["level_paths"],
        "scenario": car_fuels_scenario["level_paths"],
    },
    "liquid_fuels": weekly_hicp_paths["liquid_fuels"],
    "gas": gas_hicp,
    "electricity": electricity_hicp,
    "heat_energy": heat_hicp,
    "solid_fuels": solid_hicp,
}

aggregate_dates = common_monthly_dates(list(component_objects.values()))

baseline_unpaired = {
    name: align_paths(obj, "baseline", aggregate_dates)
    for name, obj in component_objects.items()
}
scenario_unpaired = {
    name: align_paths(obj, "scenario", aggregate_dates)
    for name, obj in component_objects.items()
}

baseline_components, final_pair_indices = independent_draw_pairing(
    baseline_unpaired,
    n_draws=N_AGG_DRAWS,
    seed=PAIRING_SEED + 1,
)

scenario_components = {
    name: scenario_unpaired[name][final_pair_indices[name]]
    for name in baseline_components
}

energy_baseline = aggregate_draw_paths_laspeyres(
    model_history["component_history"],
    baseline_components,
    aggregate_dates,
    weights,
    indices["hicp_energy"],
)

energy_scenario = aggregate_draw_paths_laspeyres(
    model_history["component_history"],
    scenario_components,
    aggregate_dates,
    weights,
    indices["hicp_energy"],
)

energy_baseline_yoy = draw_yoy_and_contributions(
    energy_baseline,
    model_history["energy"],
)
energy_scenario_yoy = draw_yoy_and_contributions(
    energy_scenario,
    model_history["energy"],
)

print(
    f"HICP Energy aggregate path: {aggregate_dates.min().date()} -> "
    f"{aggregate_dates.max().date()}"
)
print(f"Aggregate posterior draws: {energy_baseline['level_paths'].shape[0]}")
display(
    energy_baseline["weight_year_used"]
    .rename("Eurostat weight year used")
    .to_frame()
)



## 9. HICP Energy forecast fan

The median is computed **after** draw-wise aggregation. It is therefore not a weighted average of component medians.


In [ ]:

energy_level_fan = summarise_draw_paths(
    energy_baseline["level_paths"],
    aggregate_dates,
)
energy_yoy_fan = summarise_draw_paths(
    energy_baseline_yoy["yoy_paths"],
    aggregate_dates,
)

display(energy_level_fan)
display(energy_yoy_fan)

actual_energy = indices["hicp_energy"].dropna()

fig, ax = plt.subplots(figsize=(11.5, 4.8))
history = actual_energy.iloc[-72:]
ax.plot(history.index, history.to_numpy(), label="Published HICP Energy")
ax.fill_between(
    aggregate_dates,
    energy_level_fan["q05"].to_numpy(),
    energy_level_fan["q95"].to_numpy(),
    alpha=0.18,
    label="90% posterior interval",
)
ax.fill_between(
    aggregate_dates,
    energy_level_fan["q16"].to_numpy(),
    energy_level_fan["q84"].to_numpy(),
    alpha=0.28,
    label="68% posterior interval",
)
ax.plot(
    aggregate_dates,
    energy_level_fan["q50"].to_numpy(),
    linewidth=1.8,
    label="Aggregate posterior median",
)
ax.axvline(actual_energy.index[-1], linestyle="--", linewidth=1.0)
ax.set_title("HICP Energy index — aggregate posterior forecast")
ax.set_ylabel("Index")
ax.grid(alpha=0.22)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
plt.show()

actual_yoy = 100.0 * (actual_energy / actual_energy.shift(12) - 1.0)

fig, ax = plt.subplots(figsize=(11.5, 4.8))
history_yoy = actual_yoy.dropna().iloc[-72:]
ax.plot(history_yoy.index, history_yoy.to_numpy(), label="Published HICP Energy YoY")
ax.fill_between(
    aggregate_dates,
    energy_yoy_fan["q05"].to_numpy(),
    energy_yoy_fan["q95"].to_numpy(),
    alpha=0.18,
    label="90% posterior interval",
)
ax.fill_between(
    aggregate_dates,
    energy_yoy_fan["q16"].to_numpy(),
    energy_yoy_fan["q84"].to_numpy(),
    alpha=0.28,
    label="68% posterior interval",
)
ax.plot(
    aggregate_dates,
    energy_yoy_fan["q50"].to_numpy(),
    linewidth=1.8,
    label="Aggregate posterior median",
)
ax.axhline(0.0, linewidth=0.8)
ax.axvline(actual_energy.index[-1], linestyle="--", linewidth=1.0)
ax.set_title("HICP Energy inflation — aggregate posterior forecast")
ax.set_ylabel("Year-on-year percent")
ax.grid(alpha=0.22)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
plt.show()



## 10. Exact component contributions

For Laspeyres term \(T_{i,t}\) and aggregate index \(A_t\),

\[
Contribution_{i,t}
=
100\frac{T_{i,t}-T_{i,t-12}}{A_{t-12}}.
\]

Hence, draw by draw,

\[
\sum_i Contribution_{i,t}
=
100\left(\frac{A_t}{A_{t-12}}-1\right).
\]

This definition stays additive through January weight changes.


In [ ]:

contribution_paths = np.asarray(
    energy_baseline_yoy["contribution_paths"],
    dtype=float,
)
contribution_names = list(energy_baseline_yoy["components"])

median_contributions = pd.DataFrame(
    np.nanmedian(contribution_paths, axis=0),
    index=aggregate_dates,
    columns=contribution_names,
)
median_contributions["HICP Energy YoY median"] = np.nanmedian(
    energy_baseline_yoy["yoy_paths"],
    axis=0,
)
median_contributions["sum of median contributions"] = (
    median_contributions[contribution_names].sum(axis=1, min_count=1)
)

# The equality that matters is draw-wise. This is the hard numerical guard.
draw_additivity_error = np.asarray(
    energy_baseline_yoy["additivity_error"],
    dtype=float,
)
finite_additivity = np.isfinite(draw_additivity_error)
max_additivity_error = (
    float(np.max(np.abs(draw_additivity_error[finite_additivity])))
    if finite_additivity.any()
    else np.nan
)
print(f"Maximum draw-wise contribution additivity error: {max_additivity_error:.3e}")
assert not np.isfinite(max_additivity_error) or max_additivity_error < 1e-10

display(median_contributions)

fig, ax = plt.subplots(figsize=(11.5, 5.2))
for column in contribution_names:
    ax.plot(
        median_contributions.index,
        median_contributions[column],
        label=column,
    )
ax.plot(
    median_contributions.index,
    median_contributions["HICP Energy YoY median"],
    linewidth=2.2,
    linestyle="--",
    label="HICP Energy YoY median",
)
ax.axhline(0.0, linewidth=0.8)
ax.set_title("Median component contributions to HICP Energy inflation")
ax.set_ylabel("Percentage points")
ax.grid(alpha=0.22)
ax.legend(frameon=False, ncol=2)
fig.tight_layout()
plt.show()



## 11. Baseline versus tax scenario

The tax scenario is an **ex-post accounting scenario**. It never changes the pre-tax BVAR draws.

The scenario impact is calculated from matched aggregate draws:

\[
\Delta\pi_t^{tax}
=
\pi_{t,\ scenario}^{Energy}
-
\pi_{t,\ baseline}^{Energy}.
\]


In [ ]:

scenario_active = any(value is not None for value in TAX_SCENARIOS.values())

scenario_level_fan = summarise_draw_paths(
    energy_scenario["level_paths"],
    aggregate_dates,
)
scenario_yoy_fan = summarise_draw_paths(
    energy_scenario_yoy["yoy_paths"],
    aggregate_dates,
)

scenario_impact_paths = (
    np.asarray(energy_scenario_yoy["yoy_paths"], dtype=float)
    - np.asarray(energy_baseline_yoy["yoy_paths"], dtype=float)
)
scenario_impact_fan = summarise_draw_paths(
    scenario_impact_paths,
    aggregate_dates,
)

if scenario_active:
    display(
        pd.concat(
            {
                "baseline_yoy": energy_yoy_fan,
                "scenario_yoy": scenario_yoy_fan,
                "scenario_impact_pp": scenario_impact_fan,
            },
            axis=1,
        )
    )

    fig, ax = plt.subplots(figsize=(11.5, 4.8))
    ax.plot(
        aggregate_dates,
        energy_yoy_fan["q50"].to_numpy(),
        label="Baseline median",
    )
    ax.plot(
        aggregate_dates,
        scenario_yoy_fan["q50"].to_numpy(),
        linestyle="--",
        label="Tax-scenario median",
    )
    ax.fill_between(
        aggregate_dates,
        scenario_impact_fan["q16"].to_numpy(),
        scenario_impact_fan["q84"].to_numpy(),
        alpha=0.20,
        label="Scenario impact 68% interval",
    )
    ax.set_title("HICP Energy inflation — baseline versus tax scenario")
    ax.set_ylabel("Percent / percentage-point impact")
    ax.grid(alpha=0.22)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()
else:
    print("No tax scenario is active. Scenario and baseline aggregate paths are identical.")



## 12. Save the aggregate result for the dashboard

The output contains draw-wise HICP Energy levels, YoY inflation, exact contributions, the baseline/scenario pairing and the complete aggregation metadata.

The dashboard should read this store rather than reimplementing the chain-linking logic.


In [ ]:

aggregate_config = {
    "vintage": str(VINTAGE),
    "forecast_name": FORECAST_NAME,
    "n_aggregate_draws": int(N_AGG_DRAWS),
    "pairing_seed": int(PAIRING_SEED),
    "weekly_tax_mode_requested": WEEKLY_TAX_MODE,
    "weekly_tax_mode_effective": WEEKLY_TAX_MODE_EFFECTIVE,
    "tax_scenarios": TAX_SCENARIOS,
    "component_forecast_stores": {
        name: str(path) for name, path in FORECAST_STORES.items()
    },
    "bridge_diagnostics": {
        name: bridge.as_dict() for name, bridge in bridges.items()
    },
    "weight_policy": energy_baseline["future_weight_policy"],
    "cross_model_dependence": (
        "independent random pairing of marginal component predictive paths; "
        "this is not a joint posterior across the seven separately estimated BVARs"
    ),
    "other_transport_fuels_forecast": (
        "latest published HICP level held constant over the forecast horizon"
    ),
}

config_json = json.dumps(aggregate_config, sort_keys=True, default=str)
aggregate_run_id = hashlib.sha256(config_json.encode("utf-8")).hexdigest()[:20]

AGGREGATE_DIR = (
    RESULTS_ROOT
    / "hicp_energy_aggregate"
    / str(VINTAGE)
    / aggregate_run_id
)

if SAVE_AGGREGATE_RESULT:
    AGGREGATE_DIR.mkdir(parents=True, exist_ok=True)

    np.savez_compressed(
        AGGREGATE_DIR / "aggregate_draws.npz",
        baseline_level_paths=np.asarray(energy_baseline["level_paths"], dtype=float),
        baseline_yoy_paths=np.asarray(energy_baseline_yoy["yoy_paths"], dtype=float),
        baseline_contribution_paths=np.asarray(
            energy_baseline_yoy["contribution_paths"], dtype=float
        ),
        scenario_level_paths=np.asarray(energy_scenario["level_paths"], dtype=float),
        scenario_yoy_paths=np.asarray(energy_scenario_yoy["yoy_paths"], dtype=float),
        scenario_contribution_paths=np.asarray(
            energy_scenario_yoy["contribution_paths"], dtype=float
        ),
        scenario_impact_yoy_paths=np.asarray(scenario_impact_paths, dtype=float),
        transport_petrol_pair_indices=np.asarray(
            transport_pair_indices["hicp_petrol"], dtype=int
        ),
        transport_diesel_pair_indices=np.asarray(
            transport_pair_indices["hicp_diesel"], dtype=int
        ),
        final_car_fuels_pair_indices=np.asarray(
            final_pair_indices["car_fuels"], dtype=int
        ),
        final_liquid_fuels_pair_indices=np.asarray(
            final_pair_indices["liquid_fuels"], dtype=int
        ),
        final_gas_pair_indices=np.asarray(final_pair_indices["gas"], dtype=int),
        final_electricity_pair_indices=np.asarray(
            final_pair_indices["electricity"], dtype=int
        ),
        final_heat_energy_pair_indices=np.asarray(
            final_pair_indices["heat_energy"], dtype=int
        ),
        final_solid_fuels_pair_indices=np.asarray(
            final_pair_indices["solid_fuels"], dtype=int
        ),
    )

    metadata = {
        **aggregate_config,
        "aggregate_run_id": aggregate_run_id,
        "path_dates": [date.isoformat() for date in aggregate_dates],
        "components": list(energy_baseline["components"]),
        "weight_year_used": {
            date.isoformat(): int(year)
            for date, year in energy_baseline["weight_year_used"].items()
        },
        "historical_validation_max_errors": {
            name: float(value)
            for name, value in historical_validation["summary"]["max_abs_error"].items()
        },
        "six_model_history_max_error": float(
            model_history["summary"]["max_abs_error"]
        ),
        "maximum_drawwise_contribution_additivity_error": max_additivity_error,
        "scenario_active": bool(scenario_active),
    }
    (AGGREGATE_DIR / "metadata.json").write_text(
        json.dumps(metadata, indent=2, default=str),
        encoding="utf-8",
    )

    energy_level_fan.to_csv(AGGREGATE_DIR / "baseline_level_fan.csv")
    energy_yoy_fan.to_csv(AGGREGATE_DIR / "baseline_yoy_fan.csv")
    scenario_level_fan.to_csv(AGGREGATE_DIR / "scenario_level_fan.csv")
    scenario_yoy_fan.to_csv(AGGREGATE_DIR / "scenario_yoy_fan.csv")
    scenario_impact_fan.to_csv(AGGREGATE_DIR / "scenario_impact_yoy_fan.csv")
    median_contributions.to_csv(
        AGGREGATE_DIR / "median_component_contributions.csv"
    )
    bridge_table.to_csv(AGGREGATE_DIR / "weekly_bridge_diagnostics.csv")
    historical_validation["summary"].to_csv(
        AGGREGATE_DIR / "historical_reconstruction_summary.csv"
    )

    print(f"Saved aggregate run: {AGGREGATE_DIR}")
else:
    print("SAVE_AGGREGATE_RESULT=False: aggregate result was not written.")



## 13. Final audit

The aggregate run is acceptable only if:

- every historical reconstruction guard passed;
- the six-model historical HICP Energy reconstruction passed;
- draw-wise contributions add exactly to aggregate YoY inflation;
- all component forecasts refer to the same processed vintage;
- the effective weekly tax mode is visible in the metadata;
- any future weight carry-forward is explicitly recorded.

At this point the dashboard can consume the aggregate result store without duplicating the econometric or chain-linking code.


In [ ]:

audit = pd.Series({
    "processed vintage": VINTAGE,
    "forecast contract": FORECAST_NAME,
    "all cumulative + annual re-anchored historical guards": bool(
        historical_validation["summary"]["pass"].all()
    ),
    "six-model historical reconstruction": bool(
        model_history["summary"]["pass"]
    ),
    "maximum historical six-model index error": float(
        model_history["summary"]["max_abs_error"]
    ),
    "maximum draw-wise contribution additivity error": max_additivity_error,
    "weekly tax mode effective": WEEKLY_TAX_MODE_EFFECTIVE,
    "tax scenario active": scenario_active,
    "aggregate path start": aggregate_dates.min(),
    "aggregate path end": aggregate_dates.max(),
    "aggregate posterior draws": int(energy_baseline["level_paths"].shape[0]),
    "aggregate output directory": (
        str(AGGREGATE_DIR) if SAVE_AGGREGATE_RESULT else "not saved"
    ),
})
display(audit.to_frame("value"))

assert historical_validation["summary"]["pass"].all()
assert bool(model_history["summary"]["pass"])
assert not np.isfinite(max_additivity_error) or max_additivity_error < 1e-10

print("Notebook 10 aggregation audit: PASS")
